# Mamba3Yolo — Kaggle Paper Experiments Notebook

**Repository**: https://github.com/ShMazumder/Mamba3Yolo  
**Goal**: Produce clean, reproducible results (tables, curves, XAI figures, FPS, quantization drop) suitable for a CVPR/ICCV/MICCAI paper.

This notebook is designed for **Kaggle GPU** (P100/T4/V100).  
It supports:
- Self-contained Mamba3Yolo training
- Ablations (MIMO on/off, complex vs real, etc.)
- Medical multi-domain or COCO-style data
- Grad-CAM++ visualizations
- Post-training quantization notes
- Automatic result export for paper tables

**Author**: ready for paper writing  
**Date**: July 2026


## 1. Environment Setup & Reproducibility

In [ ]:
# Install dependencies (Kaggle usually has torch; we add the rest)
%pip install -q mamba-ssm einops thop seaborn timm opencv-python-headless 2>/dev/null || true
%pip install -q torchmetrics pycocotools 2>/dev/null || true

import os, sys, gc, time, json, random, math
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Image as IPImage
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Project root
WORK = Path("/kaggle/working")
PROJ = WORK / "Mamba3Yolo"
print(f"Working dir: {WORK}")


## 2. Load Mamba3Yolo from GitHub

In [ ]:
# Clone the official repo (or upload the zip as Kaggle dataset)
if not PROJ.exists():
    !git clone --depth 1 https://github.com/ShMazumder/Mamba3Yolo.git {PROJ}
else:
    print("Repo already present")

sys.path.insert(0, str(PROJ))
os.chdir(PROJ)
print("Current dir:", os.getcwd())
!ls -la


## 3. Verify Core Modules & Model

In [ ]:
from src.blocks.mamba3_odss import Mamba3ODSSBlock, HAS_MAMBA3, build_mamba3_odss
from src.models.mamba3yolo import build_mamba3yolo, Mamba3Yolo

print(f"Official Mamba-3 kernels available: {HAS_MAMBA3}")

# Build Tiny model
model = build_mamba3yolo("T", nc=80, is_mimo=True, d_state=64)
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Mamba3Yolo-T parameters: {n_params:,}")

# Shape test
x = torch.randn(2, 3, 320, 320, device=DEVICE)
with torch.no_grad():
    outs = model(x)
print("Output shapes:", [tuple(o.shape) for o in outs])
del x, outs
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


## 4. Experiment Configuration (Paper Settings)

Change these for full paper runs vs quick debug.


In [ ]:
# ====================== PAPER CONFIG ======================
CFG = {
    "scale": "T",                 # T / M / L
    "nc": 80,                     # 80 for COCO, adjust for medical
    "imgsz": 640,
    "epochs": 50,                 # 300 for full paper; 10-50 for Kaggle time limits
    "batch_size": 8 if DEVICE == "cuda" else 2,
    "lr": 1e-3,
    "weight_decay": 5e-2,
    "is_mimo": True,
    "d_state": 64,
    "amp": True,
    "num_workers": 2,
    "seed": SEED,
    "project": str(WORK / "runs" / "mamba3yolo"),
    "exp_name": f"paper_{datetime.now().strftime('%Y%m%d_%H%M')}",
    # Data: "dummy" | path to YOLO folder | Kaggle dataset path
    "data_root": "dummy",         # change to e.g. "/kaggle/input/coco-2017-yolo" or medical
    "max_train_samples": None,    # set e.g. 5000 for faster Kaggle runs
}

print(json.dumps(CFG, indent=2))
os.makedirs(CFG["project"], exist_ok=True)


## 5. Dataset (Dummy / COCO / Medical)

In [ ]:
class YoloFolderDataset(Dataset):
    '''Minimal YOLO-format dataset. Supports synthetic data for testing.'''
    def __init__(self, root, img_size=640, max_samples=None, is_train=True):
        self.root = Path(root)
        self.img_size = img_size
        self.img_dir = self.root / "images"
        self.lbl_dir = self.root / "labels"
        exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
        self.imgs = []
        for e in exts:
            self.imgs += list(self.img_dir.glob(e))
        self.imgs = sorted(self.imgs)
        if max_samples:
            self.imgs = self.imgs[:max_samples]
        self.synthetic = len(self.imgs) == 0
        if self.synthetic:
            print("[Dataset] No real images → using synthetic data (for shape/loop testing)")
            self.n = 64 if is_train else 16
        else:
            self.n = len(self.imgs)
            print(f"[Dataset] Found {self.n} images in {self.img_dir}")

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        if self.synthetic:
            img = torch.randn(3, self.img_size, self.img_size)
            targets = torch.zeros((0, 6))
            return img, targets
        from PIL import Image
        import torchvision.transforms as T
        path = self.imgs[idx]
        img = Image.open(path).convert("RGB")
        img = T.Compose([
            T.Resize((self.img_size, self.img_size)),
            T.ToTensor(),
        ])(img)
        lbl_path = self.lbl_dir / (path.stem + ".txt")
        targets = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) >= 5:
                        targets.append([0, float(p[0]), *map(float, p[1:5])])
        targets = torch.tensor(targets, dtype=torch.float32) if targets else torch.zeros((0, 6))
        return img, targets

def collate_fn(batch):
    imgs, tgts = zip(*batch)
    imgs = torch.stack(imgs)
    new = []
    for i, t in enumerate(tgts):
        if t.numel():
            t = t.clone(); t[:, 0] = i
            new.append(t)
    targets = torch.cat(new, 0) if new else torch.zeros((0, 6))
    return imgs, targets

# Build loaders
train_ds = YoloFolderDataset(CFG["data_root"], CFG["imgsz"], CFG["max_train_samples"], is_train=True)
val_ds   = YoloFolderDataset(CFG["data_root"], CFG["imgsz"], 32, is_train=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=0, collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 6. Training Loop (with logging for paper)

In [ ]:
class SimpleDetectionLoss(nn.Module):
    '''Placeholder loss that is differentiable and stable.
    Replace with full YOLO loss (DFL + BCE + CIoU) for real paper numbers.
    For now it lets the whole pipeline run and produce curves.
    '''
    def __init__(self, nc=80):
        super().__init__()
        self.nc = nc
    def forward(self, preds, targets):
        # preds: list of (B, C, H, W)
        device = preds[0].device
        loss = sum(p.float().pow(2).mean() for p in preds) * 0.01
        # add a small constant so loss is never zero
        return loss + torch.tensor(0.001, device=device, requires_grad=True)

def train_one_epoch(model, loader, optimizer, scaler, loss_fn, device, epoch, amp=True):
    model.train()
    total = 0.0
    n = 0
    t0 = time.time()
    for i, (imgs, targets) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=amp and device == "cuda"):
            preds = model(imgs)
            loss = loss_fn(preds, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        n += 1
        if i % 20 == 0:
            print(f"  ep{epoch} it{i}/{len(loader)} loss={loss.item():.4f}")
    return total / max(n, 1), time.time() - t0

def evaluate_proxy(model, loader, device):
    '''Proxy metric (feature norm / loss) until real mAP is plugged in.'''
    model.eval()
    total = 0.0
    n = 0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            preds = model(imgs)
            total += sum(p.float().abs().mean().item() for p in preds)
            n += 1
    return total / max(n, 1)


In [ ]:
# Build model, optim, etc.
model = build_mamba3yolo(CFG["scale"], nc=CFG["nc"], is_mimo=CFG["is_mimo"], d_state=CFG["d_state"])
model = model.to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
loss_fn = SimpleDetectionLoss(CFG["nc"])
scaler = GradScaler(enabled=CFG["amp"] and DEVICE == "cuda")

history = {"epoch": [], "train_loss": [], "val_proxy": [], "lr": [], "time": []}
best_proxy = float("inf")
save_dir = Path(CFG["project"]) / CFG["exp_name"]
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Starting training → {save_dir}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ===== MAIN TRAINING LOOP =====
for epoch in range(1, CFG["epochs"] + 1):
    train_loss, dt = train_one_epoch(
        model, train_loader, optimizer, scaler, loss_fn, DEVICE, epoch, CFG["amp"]
    )
    val_proxy = evaluate_proxy(model, val_loader, DEVICE)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_proxy"].append(val_proxy)
    history["lr"].append(lr)
    history["time"].append(dt)

    print(f"Epoch {epoch:03d}/{CFG['epochs']} | loss={train_loss:.4f} | val_proxy={val_proxy:.4f} | lr={lr:.2e} | {dt:.1f}s")

    # checkpoint
    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "history": history,
        "cfg": CFG,
    }
    torch.save(ckpt, save_dir / "last.pt")
    if val_proxy < best_proxy:
        best_proxy = val_proxy
        torch.save(ckpt, save_dir / "best.pt")
        print("  → new best")

# Save history
with open(save_dir / "history.json", "w") as f:
    json.dump(history, f, indent=2)
print("Training finished. History & checkpoints saved.")


## 7. Training Curves (Paper Figures)

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["epoch"], history["train_loss"], "b-o", markersize=3)
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history["epoch"], history["val_proxy"], "g-o", markersize=3)
axes[1].set_title("Val Proxy Metric")
axes[1].set_xlabel("Epoch")

axes[2].plot(history["epoch"], history["lr"], "r-")
axes[2].set_title("Learning Rate (Cosine)")
axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

plt.tight_layout()
fig.savefig(save_dir / "training_curves.png", dpi=200, bbox_inches="tight")
fig.savefig(save_dir / "training_curves.pdf", bbox_inches="tight")
plt.show()
print(f"Saved → {save_dir / 'training_curves.png'}")


## 8. Ablation Study (MIMO / Mamba-3 components)

For the paper we need controlled ablations.  
Here we run a short ablation on the same data.


In [ ]:
def quick_train(is_mimo=True, epochs=5, tag="ablation"):
    '''Short run for ablation table.'''
    m = build_mamba3yolo("T", nc=CFG["nc"], is_mimo=is_mimo, d_state=CFG["d_state"]).to(DEVICE)
    opt = optim.AdamW(m.parameters(), lr=CFG["lr"])
    scal = GradScaler(enabled=CFG["amp"] and DEVICE=="cuda")
    lf = SimpleDetectionLoss(CFG["nc"])
    losses = []
    for ep in range(1, epochs+1):
        loss, _ = train_one_epoch(m, train_loader, opt, scal, lf, DEVICE, ep, CFG["amp"])
        losses.append(loss)
    n_params = sum(p.numel() for p in m.parameters())
    # FPS proxy
    m.eval()
    x = torch.randn(1, 3, CFG["imgsz"], CFG["imgsz"], device=DEVICE)
    # warmup
    for _ in range(10):
        with torch.no_grad():
            _ = m(x)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    N = 50
    with torch.no_grad():
        for _ in range(N):
            _ = m(x)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    fps = N / (time.time() - t0)
    del m, x
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {
        "tag": tag,
        "is_mimo": is_mimo,
        "params": n_params,
        "final_loss": losses[-1],
        "fps": round(fps, 1),
    }

print("Running ablations (short)...")
results = []
results.append(quick_train(is_mimo=True,  epochs=3, tag="Mamba3-MIMO"))
results.append(quick_train(is_mimo=False, epochs=3, tag="Mamba3-SISO"))

ablation_df = pd.DataFrame(results)
print(ablation_df.to_markdown(index=False))
ablation_df.to_csv(save_dir / "ablation_mimo.csv", index=False)
ablation_df.to_latex(save_dir / "ablation_mimo.tex", index=False, float_format="%.3f")
print(f"\nSaved ablation table → {save_dir}")


## 9. Explainability (Grad-CAM++ for paper figures)

In [ ]:
from src.xai.gradcam import GradCAMPlusPlus, overlay_heatmap

# Reload best model
ckpt = torch.load(save_dir / "best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# Pick a target layer (last stage or neck)
# For demo we use the whole model and a simple forward
print("XAI module loaded. For real Grad-CAM++ on medical images:")
print("  1. Point data_root to a medical YOLO folder")
print("  2. Select a concrete target layer, e.g. model.stage3 or model.neck2")
print("  3. Call GradCAMPlusPlus(model, [target_layer])")
print("  4. Overlay and save high-res figures for the paper")

# Example synthetic visualization
x = torch.randn(1, 3, 320, 320, device=DEVICE)
with torch.no_grad():
    feats = model.get_feature_maps(x) if hasattr(model, "get_feature_maps") else None
print("Feature map keys:", list(feats.keys()) if feats else "N/A")


## 10. Quantization Experiment (for edge deployment table)

In [ ]:
from src.quant.ptq import quantize_model_dynamic, export_onnx_int8

# Dynamic quant (quick)
model_cpu = build_mamba3yolo("T", nc=CFG["nc"], is_mimo=CFG["is_mimo"]).cpu().eval()
model_cpu.load_state_dict({k: v.cpu() for k, v in ckpt["model"].items()})

qmodel = quantize_model_dynamic(model_cpu)
print("Dynamic INT8 quantization done (Linear layers).")

# Export ONNX (FP32 first)
dummy = torch.randn(1, 3, 320, 320)
onnx_path = save_dir / "mamba3yolo_t.onnx"
try:
    torch.onnx.export(model_cpu, dummy, str(onnx_path), opset_version=17,
                      input_names=["images"], output_names=["out0","out1","out2"])
    print(f"ONNX exported → {onnx_path}")
except Exception as e:
    print("ONNX export note:", e)

# Size comparison
fp32_size = sum(p.numel() * 4 for p in model_cpu.parameters()) / 1e6
print(f"Approx FP32 size: {fp32_size:.1f} MB")
print("For real W8A8 / TensorRT numbers, use the exported ONNX with trtexec or ORT quantizer.")


## 11. Paper-Ready Summary & Export

In [ ]:
# Collect everything for the paper
summary = {
    "model": f"Mamba3Yolo-{CFG['scale']}",
    "params": sum(p.numel() for p in model.parameters()),
    "mimo": CFG["is_mimo"],
    "epochs_trained": CFG["epochs"],
    "final_train_loss": history["train_loss"][-1] if history["train_loss"] else None,
    "best_val_proxy": best_proxy,
    "device": DEVICE,
    "has_mamba3_kernels": HAS_MAMBA3,
    "ablation": results,
    "save_dir": str(save_dir),
    "timestamp": datetime.now().isoformat(),
}

with open(save_dir / "paper_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("="*60)
print("PAPER EXPERIMENT SUMMARY")
print("="*60)
for k, v in summary.items():
    if k != "ablation":
        print(f"{k:25s}: {v}")
print("\nAblation table:")
print(ablation_df.to_markdown(index=False))
print("\nAll artifacts saved in:")
print(save_dir)
!ls -lh {save_dir}


## 12. Next Steps for Full Paper Numbers

1. **Real data**:  
   - Upload COCO YOLO format or medical datasets (Kvasir, BCCD, etc.) as Kaggle Datasets.  
   - Set `CFG["data_root"]` and increase `epochs` to 100–300.

2. **Real mAP**: Replace `SimpleDetectionLoss` + `evaluate_proxy` with Ultralytics validator or `torchmetrics` Detection metrics + COCO API.

3. **Full ablations**: Run the four combinations (SISO/MIMO × real/complex) for longer and report mAP / AP_s / FPS / size.

4. **Medical**: Point to multi-domain medical folders and report per-domain mAP.

5. **XAI figures**: Generate Grad-CAM++ on real medical images and insert into the paper.

6. **Edge table**: Run TensorRT INT8 on the exported ONNX and measure latency/power on Jetson or Kaggle GPU.

7. Download the whole `runs/` folder from Kaggle and use the CSVs / PNGs / TEXs directly in Overleaf.

---

**Notebook ready for paper writing.**  
Good luck with the submission!
